#### 문항 1. 네이버 VIBE Top 100 수집

네이버 VIBE 차트에서 오늘의 Top 100 순위를 수집하시오.

* 대상: https://vibe.naver.com/chart

* 추출 필드: 순위 / 곡명 / 아티스트

* 아티스트는 리스트로 담을 것 (협업곡은 여러 명)

* Selenium 사용 금지

* 100건이 모두 수집되었는지 건수로 확인할 것

* 결과를 vibe_top100.csv로 저장할 것

결과 예시
```
[{'순위': 1, '곡명': 'All I Want for Christmas Is You', '아티스트': ['Mariah Carey']},
 {'순위': 2, '곡명': 'APT.', '아티스트': ['로제 (ROSÉ)', 'Bruno Mars']},
 ...]
```

In [ ]:
import csv
import requests

API_URL = "https://apis.naver.com/vibeWeb/musicapiweb/vibe/v1/chart/track/total"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json",
    "Referer": "https://vibe.naver.com/chart",
    "Origin": "https://vibe.naver.com",
}


def fetch_top100() -> list[dict]:
    """API를 호출해서 Top 100 곡 정보를 리스트[dict]로 반환"""
    params = {
        "start": 1,
        "display": 100,  
    }
    res = requests.get(API_URL, headers=HEADERS, params=params, timeout=10)
    res.raise_for_status()

    data = res.json()
    tracks = data["response"]["result"]["chart"]["items"]["tracks"]

    results = []
    for item in tracks:
        rank = int(item["rank"]["currentRank"])
        title = item["trackTitle"]
        artists = [a["artistName"] for a in item["artists"]]
        results.append({"순위": rank, "곡명": title, "아티스트": artists})

    results.sort(key=lambda x: x["순위"])
    return results


def save_csv(rows: list[dict], filename: str = "vibe_top100.csv") -> None:
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["순위", "곡명", "아티스트"])
        for row in rows:
            artist_str = ", ".join(row["아티스트"])
            writer.writerow([row["순위"], row["곡명"], artist_str])


if __name__ == "__main__":
    top100 = fetch_top100()

    assert len(top100) == 100, f"100건이 아니라 {len(top100)}건만 수집됨!"
    print(f"✅ {len(top100)}건 수집 완료")

    save_csv(top100)
    print("✅ vibe_top100.csv 저장 완료")

    for row in top100[:5]:
        print(row)

✅ 100건 수집 완료
✅ vibe_top100.csv 저장 완료
{'순위': 1, '곡명': 'LOVE ATTACK', '아티스트': ['RESCENE(리센느)']}
{'순위': 2, '곡명': '갑자기', '아티스트': ['아이오아이 (I.O.I)']}
{'순위': 3, '곡명': 'REDRED', '아티스트': ['CORTIS (코르티스)']}
{'순위': 4, '곡명': "It's Me", '아티스트': ['아일릿(ILLIT)']}
{'순위': 5, '곡명': '여름아 부탁해', '아티스트': ['볼빨간사춘기']}


#### 문항 2. 삼성전자 일별 시세 1년치 수집

네이버 금융에서 삼성전자(005930)의 일별 시세를 1년치 수집하시오.

* 대상: https://finance.naver.com/item/sise.naver?code=005930

* 추출 필드: 날짜 / 종가 / 전일비 / 시가 / 고가 / 저가 / 거래량

* 1년치 전량을 페이지네이션으로 수집할 것

* 숫자는 정제하여 숫자로 변환할 것

* 마지막 페이지를 자동으로 감지해 종료할 것

* 결과를 samsung_1y.csv로 저장할 것

결과 예시
```
[{'날짜': '2024.12.24', '종가': 54400, '전일비': '상승 900',
  '시가': 53700, '고가': 54500, '저가': 53600, '거래량': 11559385}, ...]
```  

In [ ]:
import re
import csv
import time
from datetime import datetime, timedelta

import requests
from bs4 import BeautifulSoup

CODE = "005930" 
BASE_URL = "https://finance.naver.com/item/sise_day.naver"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Referer": "https://finance.naver.com/item/sise.naver?code=005930",
}


def clean_number(text: str) -> int:
    return int(text.replace(",", "").strip())


def parse_diff(td) -> str:
    img = td.find("img")
    direction = img["alt"] if img and img.has_attr("alt") else ""
    value = td.get_text(strip=True).replace(",", "")
    return f"{direction} {value}".strip()


def get_soup(page: int) -> BeautifulSoup:
    res = requests.get(
        BASE_URL, headers=HEADERS, params={"code": CODE, "page": page}, timeout=10
    )
    res.raise_for_status()
    res.encoding = "euc-kr"
    return BeautifulSoup(res.text, "html.parser")


def get_last_page(soup: BeautifulSoup) -> int:
    pgrr = soup.find("td", class_="pgRR")
    if pgrr is None or pgrr.find("a") is None:
        return 1
    href = pgrr.find("a")["href"]
    match = re.search(r"page=(\d+)", href)
    return int(match.group(1)) if match else 1


def parse_rows(soup: BeautifulSoup) -> list[dict]:
    rows = []
    for tr in soup.select("table.type2 tr"):
        cols = tr.find_all("td")
        if len(cols) != 7:
            continue
        date_text = cols[0].get_text(strip=True)
        if not date_text:
            continue

        rows.append({
            "날짜": date_text,
            "종가": clean_number(cols[1].get_text(strip=True)),
            "전일비": parse_diff(cols[2]),
            "시가": clean_number(cols[3].get_text(strip=True)),
            "고가": clean_number(cols[4].get_text(strip=True)),
            "저가": clean_number(cols[5].get_text(strip=True)),
            "거래량": clean_number(cols[6].get_text(strip=True)),
        })
    return rows


def fetch_one_year() -> list[dict]:
    cutoff = datetime.today() - timedelta(days=365)

    first_soup = get_soup(1)
    site_last_page = get_last_page(first_soup)

    all_rows = []
    page = 1
    soup = first_soup

    while page <= site_last_page:
        rows = parse_rows(soup)

        if not rows:
            break

        reached_cutoff = False
        for row in rows:
            row_date = datetime.strptime(row["날짜"], "%Y.%m.%d")
            if row_date < cutoff:
                reached_cutoff = True
                break
            all_rows.append(row)

        if reached_cutoff:
            break

        page += 1
        if page > site_last_page:
            break

        time.sleep(0.3) 
        soup = get_soup(page)

    return all_rows


def save_csv(rows: list[dict], filename: str = "samsung_1y.csv") -> None:
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f, fieldnames=["날짜", "종가", "전일비", "시가", "고가", "저가", "거래량"]
        )
        writer.writeheader()
        writer.writerows(rows)


if __name__ == "__main__":
    data = fetch_one_year()

    print(f"✅ 총 {len(data)}건 수집 완료 (최근 1년)")
    save_csv(data)
    print("✅ samsung_1y.csv 저장 완료")

    for row in data[:5]:
        print(row)

✅ 총 243건 수집 완료 (최근 1년)
✅ samsung_1y.csv 저장 완료
{'날짜': '2026.08.21', '종가': 280500, '전일비': '상승9500', '시가': 267000, '고가': 285000, '저가': 266000, '거래량': 25327219}
{'날짜': '2026.08.20', '종가': 271000, '전일비': '상승23500', '시가': 257000, '고가': 273000, '저가': 252500, '거래량': 26095919}
{'날짜': '2026.08.19', '종가': 247500, '전일비': '하락21000', '시가': 251500, '고가': 254500, '저가': 246500, '거래량': 22788552}
{'날짜': '2026.08.18', '종가': 268500, '전일비': '하락6000', '시가': 283000, '고가': 288000, '저가': 265000, '거래량': 24464621}
{'날짜': '2026.08.14', '종가': 274500, '전일비': '상승6500', '시가': 275000, '고가': 275500, '저가': 266000, '거래량': 21669476}


#### 문항 3. 네이버 뉴스 검색기 함수 만들기 (스크롤 포함) - 난이도 상

키워드를 받아 네이버 뉴스 검색 결과를 수집하는 함수를 작성하시오.

* 대상: https://search.naver.com/search.naver?ssc=tab.news.all

* crawl_naver_news(keyword, page) — 아래로 스크롤해야 나오는 결과까지 수집, ※page 수 만큼 스크롤

* 추출 필드: 제목 / 언론사 / 링크 / 요약

* Selenium 사용 금지. 스크롤이 유발하는 요청을 Network에서 찾아 재현할 것

* 결과를 news_{keyword}.csv로 저장할 것

결과 예시
```
[{'제목': '뉴욕증시, CPI 예상 부합에 상승 출발…나스닥 0.9%↑새 창 열림',
  '언론사': '조선비즈',
  '링크': 'https://biz.chosun.com/international/international_general/2026/08/12/N3YG7LVYFJH2FHXQP2RRFUFNS4/?utm_source=naver&utm_medium=original&utm_campaign=biz',
  '요약': '미국의 인플레이션 지표가 시장 예상에 부합하고 인공지능(AI) 관련 기업들의 실적 호조가 이어지면서 뉴욕증시가 상승 출발했다. 12일(현지시각) 뉴욕증권거래소(NYSE)에 따르면 이날 다우존스30산업평균지수는 전장보다 5.6포인트(0.01%) 오른 5만 3797.47에 거래를 시작했다. 스탠더드앤드푸어스(S&P)... '},
 {'제목': '‘하나의 영혼’이 아리스토텔레스 우정 명언? AI시대, 원전을 펼쳐라[...새 창 열림',
  '언론사': '동아일보',
  '링크': 'https://www.donga.com/news/Opinion/article/all/20260812/134468253/2',
  '요약': '《 AI가 재생산하는 인간의 오류 인공지능(AI) 시대에는 검색 기능을 잘 활용하면 자료를 손쉽게 찾을 수 있다. 그러나 자칫 AI가 내놓은 답에 지나치게 의존하다 보면 낭패를 볼 수도 있다. 우정을 다룬 가장 유명한 철학서는 아리스토텔레스의 ‘니코마코스 윤리학’이다. 친애, 우애, 우정으로 번역되는... '}, ...]
```


In [ ]:
import re
import json
import html
import time
import csv

import requests

SEARCH_URL = "https://search.naver.com/search.naver"
MORE_URL = "https://s.search.naver.com/p/newssearch/3/api/tab/more"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Referer": "https://search.naver.com/search.naver?ssc=tab.news.all",
}

_MARK_RE = re.compile(r"</?mark>")  # 검색어 강조용 <mark> 태그


def _clean_html_text(s: str) -> str:
    """<mark> 태그 제거 + &amp; 같은 HTML 엔티티 복원 + 공백 정리."""
    if not s:
        return ""
    s = _MARK_RE.sub("", s)
    s = html.unescape(s)
    return " ".join(s.split()).strip()


def _extract_bootstrap_json(page_text: str):
    marker = "entry.bootstrap("
    idx = page_text.find(marker)
    if idx == -1:
        return None

    start = page_text.find("{", page_text.find(",", idx))
    if start == -1:
        return None

    depth = 0
    in_str = False
    esc = False
    end = None
    for i in range(start, len(page_text)):
        ch = page_text[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    end = i + 1
                    break
    if end is None:
        return None

    try:
        return json.loads(page_text[start:end])
    except json.JSONDecodeError:
        return None


def _walk_news_items(node, out):
    """JSON 트리를 재귀적으로 훑어서 templateId == 'newsItem' 인 기사 카드만 모은다."""
    if isinstance(node, dict):
        if node.get("templateId") == "newsItem":
            props = node.get("props", {})
            title = props.get("title")
            title_href = props.get("titleHref")
            source = (props.get("sourceProfile") or {}).get("title")
            content = props.get("content")
            if title and title_href:
                out.append({
                    "제목": _clean_html_text(title),
                    "언론사": _clean_html_text(source) if source else "",
                    "링크": title_href,
                    "요약": _clean_html_text(content) if content else "",
                })
        for v in node.values():
            _walk_news_items(v, out)
    elif isinstance(node, list):
        for v in node:
            _walk_news_items(v, out)


def _fetch_first_page(keyword: str) -> list[dict]:
    """검색 첫 화면 (스크롤 전, 보통 10건)"""
    params = {"ssc": "tab.news.all", "where": "news", "query": keyword, "sm": "tab_jum"}
    res = requests.get(SEARCH_URL, headers=HEADERS, params=params, timeout=10)
    res.raise_for_status()

    data = _extract_bootstrap_json(res.text)
    if data is None:
        return []
    items = []
    _walk_news_items(data, items)
    return items


def _fetch_more_page(keyword: str, start: int) -> list[dict]:
    """스크롤이 유발하는 '더보기' 요청을 그대로 재현 (start=11, 21, 31 ...)"""
    params = {
        "query": keyword,
        "ssc": "tab.news.all",
        "sm": "tab_smr",
        "sort": "0",
        "start": start,
        "nso": "so:r,p:all,a:all",
        "photo": "0",
        "field": "0",
        "pd": "-1",
    }
    res = requests.get(MORE_URL, headers=HEADERS, params=params, timeout=10)
    res.raise_for_status()

    items = []
    try:
        data = res.json()
        _walk_news_items(data, items)
        if items:
            return items
    except ValueError:
        pass

    data = _extract_bootstrap_json(res.text)
    if data is not None:
        _walk_news_items(data, items)
    return items


def crawl_naver_news(keyword: str, page: int = 1) -> list[dict]:
    """
    keyword를 검색해서, 스크롤을 page번 내린 만큼의 뉴스 결과를 모두 모아 반환한다.
    page=1 -> 첫 화면만 (약 10건)
    page=N -> 첫 화면 + 스크롤 (N-1)번 추가 로드
    """
    all_items = []
    seen_links = set()

    for item in _fetch_first_page(keyword):
        if item["링크"] not in seen_links:
            seen_links.add(item["링크"])
            all_items.append(item)

    start = 11
    for _ in range(max(0, page - 1)):
        more = _fetch_more_page(keyword, start)
        if not more:
            break
        for item in more:
            if item["링크"] not in seen_links:
                seen_links.add(item["링크"])
                all_items.append(item)
        start += 10
        time.sleep(0.5

    return all_items


def save_csv(rows: list[dict], keyword: str) -> str:
    filename = f"news_{keyword}.csv"
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["제목", "언론사", "링크", "요약"])
        writer.writeheader()
        writer.writerows(rows)
    return filename


if __name__ == "__main__":
    keyword = "AI"
    page = 3

    results = crawl_naver_news(keyword, page)
    print(f"✅ '{keyword}' 검색 결과 {len(results)}건 수집 완료")

    filename = save_csv(results, keyword)
    print(f"✅ {filename} 저장 완료")

    for row in results[:3]:
        print(row)

✅ 'AI' 검색 결과 10건 수집 완료
✅ news_AI.csv 저장 완료
{'제목': "카카오톡, '카카오AI'로 새 출발…2030년 AI 매출 1조 목표", '언론사': '뉴시스', '링크': 'https://www.newsis.com/view/NISX20260821_0003757272', '요약': '카카오가 회사를 인적분할해 카카오톡·인공지능(AI)·광고·커머스 사업을 맡는 신설법인 ‘카카오AI’를 출범시킨다. 약 5000만명이 이용하는 카카오톡을 개인의 의도와 맥락을 이해하고 필요한 서비스까지 연결하는 AI 플랫폼으로 바꿔 새로운 성장동력으로 키운다는 계획이다. 카카오는 21일...'}
{'제목': '[속보]카카오, 2개 회사로 쪼개진다···카카오 AI·카카오X로 인적분할...', '언론사': '경향신문', '링크': 'https://www.khan.co.kr/article/202608211020001', '요약': '카카오가 카카오 AI·카카오X로 인적분할한다고 21일 결의했다. 카카오는 인적분할 후 신설법인 카카오AI 대표에 정신아 현 카카오 대표이사를, 존속법인 카카오X 대표에 김도영 카카오인베스트먼트 대표 겸 CA협의체 그룹투자전략실장을 각각 내정했다. 카카오AI는 카카오톡을 중심으로 인공지능(AI)...'}
{'제목': '[속보] 카카오, 두 회사로 나뉜다…카카오AI·카카오X 인적분할', '언론사': '매경이코노미', '링크': 'https://www.mk.co.kr/article/12133027', '요약': '카카오, 두 회사로 나뉜다…카카오AI·카카오X 인적분할'}
